# Libraries

In [1]:
import json, math, threading, queue, time
from datetime import datetime
import pandas as pd
import plotly.graph_objects as go
from ipywidgets import HBox, VBox, Button, HTML, Label, ToggleButtons
import paho.mqtt.client as mqtt

UI_REFRESH_SECS = 0.1  # how often the map refreshes (seconds)
INGEST_SECONDS = 0.1 # How often we subscribe Pub Messages

# 4. Data Subscription and Visualisation

## 4.1. Subscriber Configurations

In [2]:
BROKER = "test.mosquitto.org"
PORT   = 1883
TOPIC  = "470461201/A2"  # same topic as publisher
UI_REFRESH_SECS = 0.1  # how often the map refreshes (seconds)

## 4.2. Preparing A1 Data for Integration

Loads A1 via a CSV and Prepares it for lat,long integration with A2 Facilities

In [3]:
# Load the CSV
nger_df = pd.read_csv("A1_Dataset.csv")

# Filter: grid_name = 'NEM' and non-null lat,long
aug = nger_df[
    (nger_df["grid_name"] == "NEM") &
    (nger_df["latitude"].notna()) &
    (nger_df["longitude"].notna())
]

# Select relevant columns
aug = aug[["facility_name", "state", "latitude", "longitude"]]

# results
print(f"Filtered facilities: {len(aug)}")
aug.head(10)

Filtered facilities: 2937


,facility_name,state,latitude,longitude
0,Gunning Wind Farm,NSW,-34.690562,149.427009
1,Royalla Solar Farm,ACT,-35.489793,149.143002
2,Waubra Wind Farm,VIC,-37.394639,143.638601
4,Banimboola Hydro,VIC,-36.533577,147.460346
5,Bayswater Power Station,NSW,-32.383841,150.952275
6,Bogong Power Station,VIC,-36.805788,147.228116
7,Burrendong Hydro,NSW,-32.664602,149.107938
8,Clover Hydro,VIC,-36.785294,147.220429
9,Coopers Cogeneration,SA,-34.843804,138.546753
10,Copeton Hydro,NSW,-29.904756,150.923932


In [4]:
name_col = "facility_name"   
lat_col  = "latitude"
lon_col  = "longitude"

# Create a lookup dict: { facility_name_lowercase : (lat, lon) }
name_to_coords = {
    str(row[name_col]).strip().lower(): (float(row[lat_col]), float(row[lon_col]))
    for _, row in aug.iterrows()
}


## 4.3. Decoding/ Normalizing Helpers 

State and Normalization Helpers for Live MQTT Stream

In [5]:
#Global state and helpers
msg_q = queue.Queue()
running = False
metric_mode = "power"

# Tracking structures for facility markers and duplicate suppression
state = {}     # facility_id to record
idx_of = {}    # facility_id to scatter index
seen   = set() # (facility_id, timestamp) to prevent double-counting

# Arrays backing the live trace (mutated in-place)
lats, lons, names, fuels = [], [], [], []
sizes, hovertexts, customdata = [], [], []

# JSON to Python type conversion 
def as_float(x, default=0.0):
    try:
        if x in (None, "", "null"):
            return default
        return float(x)
    except Exception:
        return default

def parse_ts(x):
    try:
        return datetime.fromisoformat(str(x).replace("Z", "+00:00"))
    except Exception:
        return None
    
#Builds Hover Text
def hover_text(rec, mode="power"):  
    if mode == "emissions":
        return (
            f"<b>{rec['facility_name']}</b><br>"
            f"Emissions (current): {rec['current_emission']:.2f} tCO₂"
        )
    # default (power): compact hover
    return (
        f"<b>{rec['facility_name']}</b><br>"
        f"Power (current): {rec['current_power']:.1f} MW<br>"
        f"Emissions (current): {rec['current_emission']:.2f} tCO₂"
    )

# Normalize JSON messages to consistent key names + Python types
def normalize_keys(d):
    """Normalize any incoming JSON message to consistent Python dict."""
    return {
        "facility_id":   str(d.get("facility_id") or d.get("id") or d.get("station_id") or ""),
        "facility_name": str(d.get("facility_name") or d.get("name") or d.get("station_name") or "Unknown"),
        "lat":           as_float(d.get("lat", d.get("latitude"))),
        "lon":           as_float(d.get("lon", d.get("longitude"))),
        "fuel":          str(d.get("fuel_type", d.get("fuel_source", "Unknown"))),
        "ts_iso":        str(d.get("timestamp") or d.get("time") or ""),
        "p_mw":          as_float(d.get("power_MW", d.get("power", 0.0))),
        "e_tco2":        as_float(d.get("emissions_tCO2", d.get("emissions", 0.0))),
        "price_per_mwh": as_float(d.get("value_price_in_$/MWh")),
        "demand_mw":     as_float(d.get("value_demand_in_MW")),
        "unit_codes":     d.get("unit_codes", []),                 
        "unit_codes_str": d.get("unit_codes_str", "—"),            
    }


## 4.4. Colour Coding and Assigning by Fuel Type

In [6]:
# Fuel colour aliases
fuel_colors = {
    "Coal": "#4B0082",
    "Natural Gas": "#FF8C00",
    "Gas": "#FF8C00",
    "Solar": "#FFD700",
    "Wind": "#1E90FF",
    "Hydro": "#00CED1",
    "Oil": "#8B0000",
    "Diesel": "#B22222",
    "Battery": "#32CD32",
    "Bioenergy": "#9ACD32",
    "Geothermal": "#FF69B4",
    "Nuclear": "#ADFF2F",
    "Unknown": "#A9A9A9"
}

##Assigning Unknown to Fuel_Types Not Covered in Colour Aliases
def color_for_fuel(fuel: str) -> str:
    if not fuel:
        return fuel_colors["Unknown"]
    for key in fuel_colors:
        if key.lower() in fuel.lower():
            return fuel_colors[key]
    return fuel_colors["Unknown"]


## 4.5. Initialising/Updating Records on Map

Record Initialization & Update Helpers (sizing, hover text, coord fix, rolling totals)

In [7]:
# Compute marker size from instantaneous power (smooth scaling + min bubble size)
def size_from_total_power(mw: float) -> float:
    return max(6.0, 2.4 * math.sqrt(max(0.0, mw)))  # smooth growth, min size

def size_from_emission(tco2: float) -> float:  # <-- NEW
    return max(6.0, 2.4 * math.sqrt(max(0.0, tco2)))


# Initialize a facility record; override coords if we have a [clean name ,(lat,lon)] match
def init_rec(nk):
    
    fname = str(nk["facility_name"]).strip().lower()
    if fname in name_to_coords:
        fixed_lat, fixed_lon = name_to_coords[fname]
        nk["lat"] = fixed_lat
        nk["lon"] = fixed_lon
    
    
    ts = parse_ts(nk["ts_iso"])
    return {
        "facility_id": nk["facility_id"],
        "facility_name": nk["facility_name"],
        "lat": nk["lat"],
        "lon": nk["lon"],
        "fuel": nk["fuel"],
        "last_ts": ts,
        "current_power": nk["p_mw"],
        "current_emission": nk["e_tco2"],
        "total_power": max(0.0, nk["p_mw"]),
        "total_emission": max(0.0, nk["e_tco2"]),
        "price_per_mwh": nk["price_per_mwh"],
        "demand_mw":     nk["demand_mw"],
        "unit_codes":      nk.get("unit_codes", []),               
        "unit_codes_str":  nk.get("unit_codes_str", "—"),          
    }

# Update an existing record; optionally accumulate to rolling totals
def update_rec(rec, nk, add_to_totals: bool):
    rec["last_ts"] = parse_ts(nk["ts_iso"]) or rec["last_ts"]
    rec["current_power"] = nk["p_mw"]
    rec["current_emission"] = nk["e_tco2"]
    rec["price_per_mwh"] = nk["price_per_mwh"]
    rec["demand_mw"]     = nk["demand_mw"]
    if add_to_totals:
        if nk["p_mw"] >= 0: rec["total_power"] += nk["p_mw"]
        if nk["e_tco2"] >= 0: rec["total_emission"] += nk["e_tco2"]



## 4.6. MQTT Subscriber Setup

##### NOTE: Please ensure MQTT Publisher is already publishing before running this(Timeout has been set)

MQTT Subscriber Setup (Connect, Subscribe, and Push Incoming Messages to Queue)

In [8]:
# MQTT setup
def on_connect(client, userdata, flags, rc):
    if rc == 0:
        client.subscribe(TOPIC, qos=0) #Subscribe to live stream topic on successful connect

def on_message(client, userdata, msg):
    try:
        payload = json.loads(msg.payload.decode("utf-8")) # Decode JSON payload
        nk = normalize_keys(payload) # Standardize 
        if not nk["facility_id"] or nk["ts_iso"] == "": # skip incomplete records
            return
        msg_q.put(nk) # Push normalized message into processing queue
    except Exception:
        pass  # ignore malformed messages

client = mqtt.Client()
client.on_connect = on_connect
client.on_message = on_message


/var/folders/tv/ly87y_cd5vzcqrw28_rhj9t40000gn/T/ipykernel_37838/220985678.py:16: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client()


## 4.7. Setting Up FigureWidget for Hover

Setting Up the Live Map FigureWidget (Interactive Hover & Realtime Marker Updates)


In [9]:
# Create the interactive map figure (markers sized by power, colored by fuel, live-updatable)
fig = go.FigureWidget(
    data=[go.Scattermap(
        lat=lats,
        lon=lons,
        mode="markers",
        marker=dict(
            size=sizes,
            color=[color_for_fuel(f) for f in fuels],
            sizemode="diameter",
            opacity=0.85
        ),
        text=hovertexts,
        hoverinfo="text",
        customdata=customdata
    )],
    layout=go.Layout(
        mapbox=dict(style="open-street-map", zoom=3),
        margin=dict(l=0, r=0, t=0, b=0),
        height=650
    ),
    
)
fig.layout.mapbox.center = dict(lat=-27.5, lon=134.0)



## 4.8. Setting up HTML for Click

Setting Up Click Interaction to Display Facility Details in HTML

In [10]:
# HTML widget that will display details when a facility marker is clicked
details = HTML("<i>Click a facility…</i>")

def handle_click(trace, points, selector):
    # Ignore if no point was clicked (e.g., click on empty map)
    if not points.point_inds:
        return
    i = points.point_inds[0]
    rec = trace.customdata[i]

    unit_codes = ", ".join(rec.get("unit_codes") or []) if rec.get("unit_codes") else rec.get("unit_codes_str", "—")  # NEW

    # Build HTML content dynamically for the selected facility
    details.value = (
        f"<h3>{rec['facility_name']}</h3>"
        f"<b>Fuel:</b> {rec['fuel']}<br>"
        f"<b>Current Power:</b> {rec['current_power']:.1f} MW<br>"
        f"<b>Current Emissions:</b> {rec['current_emission']:.2f} tCO₂<br>"
        f"<b>Price:</b> {rec.get('price_per_mwh', 0.0):.2f} $/MWh<br>"
        f"<b>Demand:</b> {rec.get('demand_mw', 0.0):.1f} MW<br>"
        f"<b>Total Power:</b> {rec['total_power']:.1f} MW<br>"
        f"<b>Total Emissions:</b> {rec['total_emission']:.2f} tCO₂<br>"
        #static units block (click-only) 
        f"<div style='margin-top:10px;'>"
        f"  <div style='font-size:11px;color:#64748b;text-transform:uppercase;letter-spacing:.02em;margin-bottom:6px;'>Unit Codes</div>"
        f"  <div style='font-family:ui-monospace, SFMono-Regular, Menlo, monospace; "
        f"              font-size:13px;background:#f8fafc;border:1px dashed #cbd5e1; "
        f"              padding:8px 10px;border-radius:10px;'>{unit_codes}</div>"
        f"</div>"
    )

# Attach the click callback to the map markers
fig.data[0].on_click(handle_click)



## 4.9. Actual ingestion

Live Ingestion Loop: Update State and Marker Data Without Redrawing the Figure

In [11]:
# Ingest loop (state update, no redraw)
def ingest_loop():
    while running:
        try:
            nk = msg_q.get(timeout=0.25)
        except queue.Empty:
            continue

        fid = nk["facility_id"]
        key = (fid, nk["ts_iso"])
        add_to_totals = key not in seen
        if add_to_totals:
            seen.add(key)

        if fid not in state:
            rec = init_rec(nk)
            state[fid] = rec

            # Append new point
            idx = len(lats)
            idx_of[fid] = idx
            lats.append(rec["lat"])
            lons.append(rec["lon"])
            names.append(rec["facility_name"])
            fuels.append(rec["fuel"])

            #size & hover by mode
            if metric_mode == "emissions":
                sizes.append(size_from_emission(rec["current_emission"]))
                hovertexts.append(hover_text(rec, "emissions"))
            else:
                sizes.append(size_from_total_power(rec["total_power"]))
                hovertexts.append(hover_text(rec, "power"))

            customdata.append(rec.copy())

            # Update colours when a new facility appears
            fig.data[0].marker.color = [color_for_fuel(f) for f in fuels]

        else:
            rec = state[fid]
            update_rec(rec, nk, add_to_totals)
            i = idx_of[fid]

            # size & hover by mode
            if metric_mode == "emissions":
                sizes[i] = size_from_emission(rec["current_emission"])
                hovertexts[i] = hover_text(rec, "emissions")
            else:
                sizes[i] = size_from_total_power(rec["total_power"])
                hovertexts[i] = hover_text(rec, "power")

            customdata[i] = rec.copy()  # keep click info synced
        time.sleep(INGEST_SECONDS)

In [12]:
#Render loop
def render_loop():
    while running:
        with fig.batch_update():
            fig.data[0].lat = tuple(lats)
            fig.data[0].lon = tuple(lons)
            fig.data[0].marker.size = tuple(sizes)
            fig.data[0].text = tuple(hovertexts)
            fig.data[0].customdata = tuple(customdata)
        status.value = f"running · facilities: {len(lats)}"
        time.sleep(UI_REFRESH_SECS)


## 4.10 Display

Controls and Display: Start/Stop Buttons and Threaded Live Updating

In [13]:
# Controls and display
start_btn = Button(description="Start", button_style="success")
stop_btn  = Button(description="Stop",  button_style="warning")
status    = Label("stopped")
metric_tg = ToggleButtons(                      # <-- NEW
    options=[("Power","power"), ("Emissions","emissions")],
    value="power",
    description="Metric:"
)

ingest_thr = None
render_thr = None

def apply_metric():                              # <-- NEW
    """Recompute sizes & hovers for current metric without resetting anything."""
    if not customdata:
        return
    if metric_mode == "emissions":
        for i, rec in enumerate(customdata):
            sizes[i] = size_from_emission(rec["current_emission"])
            hovertexts[i] = hover_text(rec, "emissions")
    else:
        for i, rec in enumerate(customdata):
            sizes[i] = size_from_total_power(rec["total_power"])
            hovertexts[i] = hover_text(rec, "power")

    # Push to figure (no redraw rebuild)
    with fig.batch_update():
        fig.data[0].marker.size = tuple(sizes)
        fig.data[0].text = tuple(hovertexts)

#Handles Toggle
def on_metric_change(change):                    
    global metric_mode
    if change["name"] == "value":
        metric_mode = change["new"]
        apply_metric()
metric_tg.observe(on_metric_change, names="value")  # Change from E.g. Power to Emission

def start(_=None):
    global running, ingest_thr, render_thr
    if running:
        return
    running = True
    client.connect(BROKER, PORT, keepalive=60)
    client.loop_start()
    ingest_thr = threading.Thread(target=ingest_loop, daemon=True)
    render_thr = threading.Thread(target=render_loop, daemon=True)
    ingest_thr.start(); render_thr.start()
    status.value = "starting…"

def stop(_=None):
    global running
    if not running:
        return
    running = False
    try:
        client.loop_stop()
        client.disconnect()
    except Exception:
        pass
    status.value = "stopped"

start_btn.on_click(start)
stop_btn.on_click(stop)

fig.layout.dragmode = "zoom"

# Display controls, map, and details
display(
    VBox([
        HBox([start_btn, stop_btn, status, metric_tg]),   
        HBox([fig, details])
    ])
)

## 4.11 Reset 

Resetting State, UI, and MQTT Connection

In [27]:
# Reset state and UI 
#Retains (lat,long) coordinates 
running = False

state.clear() # facility state records
idx_of.clear() # map index lookup
seen.clear() # timestamp dedupe memory

# Clear live map backing arrays
lats.clear()
lons.clear()
names.clear()
fuels.clear()
sizes.clear()
hovertexts.clear()
customdata.clear()


# Stop MQTT client if running
try:
    client.loop_stop()
    client.disconnect()
except:
    pass

# Clear figure widget data fully
fig.data[0].lat = []
fig.data[0].lon = []
fig.data[0].marker.size = []
fig.data[0].text = []
fig.data[0].customdata = []

# Update status label
status.value = "Reset (ready to start again)"
print("Reset complete. Click to start again.")



Reset complete. Click to start again.
